In [ ]:
import shutil
import os

OUTPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("✅ Output folder cleared.")

✅ Output folder cleared.


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

INPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/04_Structured_LightCurves"

OUTPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

In [ ]:
import os

print(len(os.listdir(INPUT_FOLDER)))

1172


In [ ]:
import pandas as pd
import os

OUTPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

# Create 20 tiny test files
for i in range(20):
    pd.DataFrame({"a":[1,2,3]}).to_csv(
        os.path.join(OUTPUT_FOLDER, f"test_{i}.csv"),
        index=False
    )

print(len([
    f for f in os.listdir(OUTPUT_FOLDER)
    if f.endswith(".csv")
]))

20


In [ ]:
import shutil
import os

LOCAL_OUTPUT = "/content/Preprocessed_LightCurves"

if os.path.exists(LOCAL_OUTPUT):
    shutil.rmtree(LOCAL_OUTPUT)

os.makedirs(LOCAL_OUTPUT)

print("✅ Local folder ready")

✅ Local folder ready


In [2]:
import os
import time
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.signal import savgol_filter

# ==========================================================
# PATHS
# ==========================================================

INPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/04_Structured_LightCurves"


OUTPUT_FOLDER ="/content/Preprocessed_LightCurves"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ==========================================================
# SAVITZKY-GOLAY DETRENDING
# ==========================================================

def savgol_detrend_sector(df_sector,
                          window_length=1001,
                          polyorder=2):

    sector = df_sector.copy()

    flux = sector["Flux"].values

    if len(flux) <= window_length:
        window_length = len(flux) - 1

    if window_length % 2 == 0:
        window_length -= 1

    if window_length < 5:

        baseline = flux.copy()

        sector["Baseline"] = baseline

        sector["Normalized_Flux"] = np.ones_like(flux)

        sector["Normalized_Flux_Error"] = (
            sector["Flux_Error"].values / baseline
        )

        sector["Detrended_Flux"] = np.zeros_like(flux)

        return sector

    baseline = savgol_filter(
        flux,
        window_length=window_length,
        polyorder=polyorder
    )

    # Prevent division by zero
    baseline = np.where(baseline == 0, np.nan, baseline)

    sector["Baseline"] = baseline

    # ----------- THIS IS THE IMPORTANT PART -------------

    sector["Normalized_Flux"] = (
        flux / baseline
    )

    sector["Normalized_Flux_Error"] = (
        sector["Flux_Error"].values / baseline
    )

    sector["Detrended_Flux"] = (
        (flux - baseline) / baseline
    )

    return sector

# ==========================================================
# PREPROCESS ONE STAR
# ==========================================================

def preprocess_star(df):

    df = df.dropna(
        subset=[
            "Time",
            "Flux",
            "Flux_Error"
        ]
    )

    df = df[
        df["Quality"] == 0
    ].copy()

    sectors = []

    for sector in sorted(df["Sector"].unique()):

        temp = df[
            df["Sector"] == sector
        ]

        temp = savgol_detrend_sector(temp)

        sectors.append(temp)

    return pd.concat(
        sectors,
        ignore_index=True
    )

# ==========================================================
# PROCESS ALL STARS
# ==========================================================

files = sorted([
    f for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".csv")
])

success = 0
failed = []

print("="*60)
print("PREPROCESSING STARTED")
print("="*60)

for file in tqdm(files):

    try:

        input_path = os.path.join(
            INPUT_FOLDER,
            file
        )

        df = pd.read_csv(input_path)

        final_df = preprocess_star(df)

        output_path = os.path.join(
            OUTPUT_FOLDER,
            file
        )

        saved = False

        # --------------------------------------------------
        # Retry saving up to 3 times
        # --------------------------------------------------

        for attempt in range(3):

            try:

                with open(output_path, "w") as f:

                    final_df.to_csv(
                        f,
                        index=False
                    )

                    f.flush()

                    os.fsync(f.fileno())

                if os.path.exists(output_path):

                    saved = True

                    break

            except Exception:

                time.sleep(1)

        if not saved:

            raise RuntimeError(
                "Could not save file."
            )

        success += 1

    except Exception as e:

        failed.append(file)

        print(f"\nFailed : {file}")

        print(e)

print()

print("="*60)
print("PREPROCESSING COMPLETE")
print("="*60)

print("Successful :", success)

print("Failed :", len(failed))

print()

print(
    "CSV Files Saved :",
    len([
        f
        for f in os.listdir(OUTPUT_FOLDER)
        if f.endswith(".csv")
    ])
)

if len(failed):

    print("\nFailed Files")

    for f in failed:

        print(f)

PREPROCESSING STARTED


100%|██████████| 1172/1172 [1:25:13<00:00,  4.36s/it]


PREPROCESSING COMPLETE
Successful : 1172
Failed : 0

CSV Files Saved : 1172


In [3]:
import os

print(len(os.listdir(OUTPUT_FOLDER)))

1172


In [4]:
import shutil
import os

DRIVE_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

if os.path.exists(DRIVE_FOLDER):
    shutil.rmtree(DRIVE_FOLDER)

os.makedirs(DRIVE_FOLDER, exist_ok=True)

print("✅ Old Drive folder deleted and recreated.")

✅ Old Drive folder deleted and recreated.


In [5]:
import os
import shutil
from tqdm import tqdm

LOCAL_FOLDER = "/content/Preprocessed_LightCurves"

DRIVE_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

files = sorted([
    f for f in os.listdir(LOCAL_FOLDER)
    if f.endswith(".csv")
])

for file in tqdm(files):

    shutil.copy2(
        os.path.join(LOCAL_FOLDER, file),
        os.path.join(DRIVE_FOLDER, file)
    )

print("✅ All files copied successfully!")

100%|██████████| 1172/1172 [07:24<00:00,  2.64it/s]

✅ All files copied successfully!


In [6]:
import os

local_count = len([
    f for f in os.listdir("/content/Preprocessed_LightCurves")
    if f.endswith(".csv")
])

drive_count = len([
    f for f in os.listdir("/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves")
    if f.endswith(".csv")
])

print("="*50)
print("Verification")
print("="*50)

print("Local CSVs :", local_count)
print("Drive CSVs :", drive_count)

if local_count == drive_count == 1172:
    print("\n✅ SUCCESS! All 1172 files copied.")
else:
    print("\n⚠️ Counts do not match.")

Verification
Local CSVs : 1172
Drive CSVs : 1172

✅ SUCCESS! All 1172 files copied.


In [7]:
import pandas as pd
import os

sample = sorted(os.listdir(DRIVE_FOLDER))[0]

df = pd.read_csv(
    os.path.join(DRIVE_FOLDER, sample)
)

print(df.columns)

df.head()

Index(['TIC_ID', 'Label', 'Sector', 'Time', 'Flux', 'Flux_Error', 'Quality',
       'Cadence', 'Baseline', 'Normalized_Flux', 'Normalized_Flux_Error',
       'Detrended_Flux'],
      dtype='object')


,TIC_ID,Label,Sector,Time,Flux,Flux_Error,Quality,Cadence,Baseline,Normalized_Flux,Normalized_Flux_Error,Detrended_Flux
0,100100829,EB,2,1354.111509,1678.6504,9.462868,0,91190,1619.154952,1.036745,0.005844,0.036745
1,100100829,EB,2,1354.112898,1664.3468,9.423312,0,91191,1619.134909,1.027923,0.005820,0.027923
2,100100829,EB,2,1354.114287,1663.1503,9.422101,0,91192,1619.114901,1.027197,0.005819,0.027197
3,100100829,EB,2,1354.117065,1647.0486,9.403674,0,91194,1619.094925,1.017265,0.005808,0.017265
4,100100829,EB,2,1354.118453,1654.6439,9.418030,0,91195,1619.074983,1.021969,0.005817,0.021969
